<a href="https://colab.research.google.com/github/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection/blob/First-branch/Pipeline_2_ADASYN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

import shap

from scipy.stats import spearmanr
from itertools import combinations

from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")

In [ ]:
# le nom du fichier CSV du dataset PaySim
DATA_PATH = "PS_20174392719_1491204439457_log.csv"
# le nom du fichier compressé téléchargé depuis Kaggle
ZIP_PATH  = "paysim1.zip"

if not os.path.exists(DATA_PATH):
    print("\n[INFO] Téléchargement du dataset via Kaggle API...")

    # Commande sert à télécharger le dataset PaySim depuis Kaggle grâce à l’API Kaggle.
    os.system("kaggle datasets download -d ealaxi/paysim1")
    if os.path.exists(ZIP_PATH):
        # Ouverture et extraction du fichier ZIP
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(".")
        print("[INFO] Extraction terminée.")

    # Cette partie s’exécute si le fichier ZIP n’a pas été trouvé après le téléchargement.
    else:
        raise FileNotFoundError(
            "Le fichier ZIP PaySim est introuvable. "
            "Vérifiez votre configuration Kaggle."
        )

df = pd.read_csv(DATA_PATH)


[INFO] Téléchargement du dataset via Kaggle API...
[INFO] Extraction terminée.


In [ ]:
CATEGORICAL_FEATURES = ['type']
NUMERICAL_FEATURES   = [
    'step', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest'
]
TARGET = 'isFraud'

X = df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"\n[SPLIT] Train : {len(X_train):,} | Test : {len(X_test):,}")
print(f"[SPLIT] Fraudes dans train : {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"[SPLIT] Fraudes dans test  : {y_test.sum():,} ({y_test.mean()*100:.4f}%)")

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print(f"Train : {X_train.shape[0]:,} observations")
print(f"Test  : {X_test.shape[0]:,} observations")

print("\nDistribution de la cible :")

print(
    f"Train - fraude : "
    f"{y_train.sum():,} "
    f"({y_train.mean()*100:.4f}%)"
)

print(
    f"Test  - fraude : "
    f"{y_test.sum():,} "
    f"({y_test.mean()*100:.4f}%)"
)


[SPLIT] Train : 5,090,096 | Test : 1,272,524
[SPLIT] Fraudes dans train : 6,570 (0.1291%)
[SPLIT] Fraudes dans test  : 1,643 (0.1291%)
TRAIN / TEST SPLIT
Train : 5,090,096 observations
Test  : 1,272,524 observations

Distribution de la cible :
Train - fraude : 6,570 (0.1291%)
Test  - fraude : 1,643 (0.1291%)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            CATEGORICAL_FEATURES
        ),
        (
            "numerical",
            StandardScaler(),
            NUMERICAL_FEATURES
        )
    ]
)

In [ ]:
INNER_CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
def get_adasyn_params(trial):
  params = {
      "sampling_strategy": trial.suggest_float("adasyn_sampling_strategy", 0.1, 1.0),
      "n_neighbors": trial.suggest_int("adasyn_n_neighbors", 5, 25),
      "random_state" : 42
  }
  return params

In [ ]:
def get_xgb_params(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators", 250, 400
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 7, 11
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.2, log=True
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.80, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 2.0, 6.0
        ),

        "eval_metric": "aucpr",
        "random_state": 42,
        "n_jobs": -1
    }

    return params

In [ ]:
def compute_shap_importance(
    model,
    # La matrice des données de validation (déjà prétraitée/encodée)
    X_validation,
    # Le nombre d'observations à extraire pour calculer les valeurs SHAP
    sample_size=500,
    random_state=42
):
    """
    Retourne un vecteur :
    [importance_feature_1, importance_feature_2, ...]
    """

    # Nombre d'observations à utiliser pour SHAP
    n_samples = min(
        sample_size,
        X_validation.shape[0] # Récupère le nombre total de lignes dans X_validation
    )

    # Échantillonnage aléatoire reproductible
    rng = np.random.RandomState(random_state)

    indices = rng.choice(
        # rng.choice() Sélectionne aléatoirement n_samples indices parmi le nombre total de lignes
        X_validation.shape[0],
        size=n_samples,
        replace=False
    )

    # Extrait les lignes sélectionnées dans la matrice X_validation pour créer le sous-ensemble X_sample
    X_sample = X_validation[indices]

    explainer = shap.TreeExplainer(model)

    shap_values = explainer.shap_values(
        X_sample
    )

    # Importance globale : moyenne de |SHAP| pour chaque variable
    mean_abs_shap = np.abs(
        shap_values
    ).mean(axis=0)
    # .mean(axis=0) : Calcule la moyenne des valeurs absolues pour chaque variable

    return mean_abs_shap

In [ ]:
def compute_spearman_stability(fold_importances):
    """
    Calcule la stabilité globale des explications SHAP.

    La stabilité correspond à la moyenne des corrélations
    de Spearman entre les importances des différents folds.
    """

    correlations = []

    # Toutes les combinaisons possibles de deux folds
    for importance_a, importance_b in combinations(
        # fold_importances : Une liste contenant les vecteurs d'importance SHAP calculés pour chaque fold lors de la cv
        fold_importances,
        2
    ):

        rho, _ = spearmanr(
            importance_a,
            importance_b
        )

        correlations.append(rho)

    # Moyenne des corrélations
    stability_score = np.mean(
        correlations
    )

    return stability_score

In [ ]:
def objective(trial):

    adasyn_params = get_adasyn_params(trial)
    xgb_params = get_xgb_params(trial)

    fold_pr_auc = []
    fold_shap_importances = []
    fold_details = []

    for fold, (train_idx, val_idx) in enumerate(
        INNER_CV.split(X_train, y_train),
        start=1
    ):

        print( f"\nTrial {trial.number} | Fold {fold}/{INNER_CV.n_splits}")

        X_tr_raw = X_train.iloc[train_idx]
        X_val_raw = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        prep = clone(preprocessor)
        prep.fit( X_tr_raw, y_tr)
        X_tr = prep.transform( X_tr_raw )
        X_val = prep.transform( X_val_raw )

        # Re-échantillonnage ADASYN avec adaptation du seed par fold
        fold_adasyn_params = adasyn_params.copy()
        fold_adasyn_params["random_state"] = 42 + fold
        adasyn = ADASYN(**fold_adasyn_params)
        X_tr_res, y_tr_res = adasyn.fit_resample(X_tr, y_tr)

        model = XGBClassifier(**xgb_params)
        model.fit(X_tr_res, y_tr_res)

        y_val_proba = model.predict_proba(
            X_val
        )[:, 1]

        pr_auc = average_precision_score(
            y_val,
            y_val_proba
        )

        fold_pr_auc.append(
            pr_auc
        )

        fold_details.append({
            "fold": fold,
            "pr_auc": float(pr_auc),
            "n_train_resampled": int(len(X_tr_res)),
            "n_validation": int(len(X_val_raw)),
            "fraud_train_resampled": int(y_tr_res.sum()),
            "fraud_validation": int(y_val.sum())
        })

        shap_importance = compute_shap_importance(
            model=model,
            X_validation=X_val,
            sample_size=500,
            random_state=42 + fold
        )

        fold_shap_importances.append(
            shap_importance
        )

        print( f"   PR-AUC = {pr_auc:.6f}" )

    mean_pr_auc = np.mean(
        fold_pr_auc
    )

    shap_stability = compute_spearman_stability(
        fold_shap_importances
    )

    print( f"\nTrial {trial.number} terminé")

    trial.set_user_attr(
        "fold_pr_auc",
        [float(x) for x in fold_pr_auc]
    )

    trial.set_user_attr(
        "mean_pr_auc",
        float(mean_pr_auc)
    )

    trial.set_user_attr(
        "shap_stability",
        float(shap_stability)
    )

    trial.set_user_attr(
        "fold_details",
        fold_details
    )

    print( f"Mean PR-AUC       = {mean_pr_auc:.6f}")

    print( f"SHAP Stability    = {shap_stability:.6f}")

    return (
        mean_pr_auc,
        shap_stability
    )

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 30.2 MB/s eta 0:00:00


In [ ]:
import optuna
from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.INFO)

# ============================================================
# SQLITE SUR GOOGLE DRIVE
# ============================================================

DB_PATH = "/content/drive/MyDrive/Optuna_Fraud_Detection_v3.db"

STORAGE = optuna.storages.RDBStorage(
    url=f"sqlite:///{DB_PATH}",

    # Heartbeat toutes les 60 secondes
    heartbeat_interval=60,

    # Si aucun heartbeat pendant 5 minutes,
    # le trial est considéré comme interrompu
    grace_period=300,

    # Évite certaines erreurs de verrouillage SQLite
    engine_kwargs={
        "connect_args": {
            "timeout": 60
        }
    }
)

STUDY_NAME = "XGBoost_PR_AUC_SHAP_Stability_ADASYN"

# ============================================================
# CREER OU RECHARGER L'ETUDE
# ============================================================

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,

    directions=[
        "maximize",   # PR-AUC
        "maximize"    # SHAP Stability
    ],

    load_if_exists=True
)

completed_trials = [
    t for t in study.trials
    if t.state == TrialState.COMPLETE
]

print("=" * 70)
print("ETUDE OPTUNA CHARGEE")
print("=" * 70)

print(f"Nom de l'étude       : {STUDY_NAME}")
print(f"Base SQLite          : {DB_PATH}")
print(f"Trials enregistrés   : {len(study.trials)}")
print(f"Trials terminés      : {len(completed_trials)}")

[I 2026-09-20 11:55:50,699] Using an existing study with `study_name='XGBoost_PR_AUC_SHAP_Stability_ADASYN'` instead of creating a new one.


ETUDE OPTUNA CHARGEE
Nom de l'étude       : XGBoost_PR_AUC_SHAP_Stability_ADASYN
Base SQLite          : /content/drive/MyDrive/Optuna_Fraud_Detection_v3.db
Trials enregistrés   : 58
Trials terminés      : 46


In [ ]:
TOTAL_TRIALS = 100

completed_trials = sum(
    t.state == TrialState.COMPLETE
    for t in study.trials
)

remaining_trials = max(
    0,
    TOTAL_TRIALS - completed_trials
)

print(f"Trials terminés actuellement : {completed_trials}")
print(f"Trials restants              : {remaining_trials}")

if remaining_trials > 0:

    study.optimize(
        objective,
        n_trials=remaining_trials
    )

else:

    print("Les 100 trials sont déjà terminés.")

Trials terminés actuellement : 46
Trials restants              : 54

Trial 58 | Fold 1/5
   PR-AUC = 0.975308

Trial 58 | Fold 2/5
   PR-AUC = 0.968030

Trial 58 | Fold 3/5
   PR-AUC = 0.978885

Trial 58 | Fold 4/5
   PR-AUC = 0.977559

Trial 58 | Fold 5/5


[I 2026-09-20 12:14:37,539] Trial 58 finished with values: [0.9750633982293195, 0.9800000000000001] and parameters: {'adasyn_sampling_strategy': 0.5490287079897948, 'adasyn_n_neighbors': 15, 'n_estimators': 385, 'max_depth': 9, 'learning_rate': 0.0999571246251812, 'subsample': 0.9179356996640189, 'gamma': 2.6708702650228875}.


   PR-AUC = 0.975535

Trial 58 terminé
Mean PR-AUC       = 0.975063
SHAP Stability    = 0.980000

Trial 59 | Fold 1/5
   PR-AUC = 0.973982

Trial 59 | Fold 2/5
   PR-AUC = 0.966900

Trial 59 | Fold 3/5
   PR-AUC = 0.980384

Trial 59 | Fold 4/5
   PR-AUC = 0.978415

Trial 59 | Fold 5/5


[I 2026-09-20 12:31:15,970] Trial 59 finished with values: [0.9749740306351743, 0.9754545454545456] and parameters: {'adasyn_sampling_strategy': 0.527173099158088, 'adasyn_n_neighbors': 19, 'n_estimators': 280, 'max_depth': 10, 'learning_rate': 0.09826733008181901, 'subsample': 0.8639406588229726, 'gamma': 2.2929193309864475}.


   PR-AUC = 0.975189

Trial 59 terminé
Mean PR-AUC       = 0.974974
SHAP Stability    = 0.975455

Trial 60 | Fold 1/5
   PR-AUC = 0.973567

Trial 60 | Fold 2/5
   PR-AUC = 0.965573

Trial 60 | Fold 3/5
   PR-AUC = 0.978091

Trial 60 | Fold 4/5
   PR-AUC = 0.975283

Trial 60 | Fold 5/5


...

   PR-AUC = 0.973056

Trial 63 terminé
Mean PR-AUC       = 0.973277
SHAP Stability    = 0.994545

Trial 64 | Fold 1/5
   PR-AUC = 0.971913

Trial 64 | Fold 2/5
   PR-AUC = 0.967727

Trial 64 | Fold 3/5
   PR-AUC = 0.977190

Trial 64 | Fold 4/5
   PR-AUC = 0.976261

Trial 64 | Fold 5/5


[I 2026-09-20 13:44:10,734] Trial 64 finished with values: [0.9735550828807191, 0.9927272727272728] and parameters: {'adasyn_sampling_strategy': 0.30933599766555553, 'adasyn_n_neighbors': 15, 'n_estimators': 264, 'max_depth': 9, 'learning_rate': 0.10491756067189074, 'subsample': 0.9503818570742866, 'gamma': 4.224831470928019}.


   PR-AUC = 0.974685

Trial 64 terminé
Mean PR-AUC       = 0.973555
SHAP Stability    = 0.992727

Trial 65 | Fold 1/5
   PR-AUC = 0.972459

Trial 65 | Fold 2/5
   PR-AUC = 0.966071

Trial 65 | Fold 3/5
   PR-AUC = 0.976166

Trial 65 | Fold 4/5
   PR-AUC = 0.975180

Trial 65 | Fold 5/5


[I 2026-09-20 13:58:49,651] Trial 65 finished with values: [0.972748036447576, 0.9881818181818183] and parameters: {'adasyn_sampling_strategy': 0.392332507835504, 'adasyn_n_neighbors': 13, 'n_estimators': 285, 'max_depth': 9, 'learning_rate': 0.07494444020918627, 'subsample': 0.9917859583186186, 'gamma': 4.864358730849017}.


   PR-AUC = 0.973864

Trial 65 terminé
Mean PR-AUC       = 0.972748
SHAP Stability    = 0.988182

Trial 66 | Fold 1/5
   PR-AUC = 0.973877

Trial 66 | Fold 2/5
   PR-AUC = 0.969131

Trial 66 | Fold 3/5
   PR-AUC = 0.978222

Trial 66 | Fold 4/5
   PR-AUC = 0.977865

Trial 66 | Fold 5/5


In [ ]:
from google.colab import files

# 1. Extraction brute des données depuis Optuna
df_results = study.trials_dataframe()

# 2. Sauvegarde au format CSV (texte brut, conserve toutes les décimales)
file_name = "optuna_study_results_ADASYN.csv"
df_results.to_csv(file_name, index=False)

# 3. Téléchargement direct sur votre ordinateur
files.download(file_name)

print(f"Fichier téléchargé avec succès : {file_name}")
print(f"Nombre total de trials exportés : {len(df_results)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Fichier téléchargé avec succès : optuna_study_results_ADASYN.csv
Nombre total de trials exportés : 58


In [ ]:
import optuna

DB_PATH = "/content/drive/MyDrive/Optuna_Fraud_Detection_v3.db"
STUDY_NAME = "XGBoost_PR_AUC_SHAP_Stability_ADASYN"

study = optuna.load_study(study_name=STUDY_NAME, storage=f"sqlite:///{DB_PATH}")

print(f"Nombre total de trials enregistrés : {len(study.trials)}")

# Affichage des détails de chaque trial enregistré
for trial in study.trials:
    print(f"Trial {trial.number} | Statut: {trial.state} | Valeurs: {trial.values}")

Nombre total de trials enregistrés : 58
Trial 0 | Statut: TrialState.COMPLETE | Valeurs: [0.970094522492517, 0.9863636363636366]
Trial 1 | Statut: TrialState.COMPLETE | Valeurs: [0.9712874989909073, 0.9627272727272727]
Trial 2 | Statut: TrialState.COMPLETE | Valeurs: [0.9740384050845735, 0.970909090909091]
Trial 3 | Statut: TrialState.FAIL | Valeurs: None
Trial 4 | Statut: TrialState.COMPLETE | Valeurs: [0.9737917950999139, 0.9927272727272728]
Trial 5 | Statut: TrialState.COMPLETE | Valeurs: [0.9714595638383361, 0.9709090909090909]
Trial 6 | Statut: TrialState.COMPLETE | Valeurs: [0.9730579466903834, 0.95]
Trial 7 | Statut: TrialState.COMPLETE | Valeurs: [0.9724574477694083, 0.9827272727272728]
Trial 8 | Statut: TrialState.COMPLETE | Valeurs: [0.9697617420827619, 0.9745454545454546]
Trial 9 | Statut: TrialState.COMPLETE | Valeurs: [0.9741254799486926, 0.9800000000000001]
Trial 10 | Statut: TrialState.COMPLETE | Valeurs: [0.9684149393029774, 0.9700000000000001]
Trial 11 | Statut: TrialS

In [ ]:
from optuna.trial import TrialState

# The variable 'completed_trials' was previously redefined as an integer.
# We need to get the actual list of completed trial objects from the study.
completed_trials_list = [
    t for t in study.trials
    if t.state == TrialState.COMPLETE
]

if completed_trials_list:
    best_trial = max(completed_trials_list, key=lambda t: t.values[0])

    print(f"Meilleur Trial pour PR-AUC : Trial {best_trial.number}")
    print(f"  - PR-AUC max     : {best_trial.values[0]:.6f}")
    print(f"  - SHAP Stability : {best_trial.values[1]:.6f}")
    print(f"  - Hyperparamètres : {best_trial.params}")
else:
    print("Aucun trial complété n'a été trouvé pour déterminer le meilleur.")

Meilleur Trial pour PR-AUC : Trial 22
  - PR-AUC max     : 0.975100
  - SHAP Stability : 0.980000
  - Hyperparamètres : {'adasyn_sampling_strategy': 0.48127163300656195, 'adasyn_n_neighbors': 13, 'n_estimators': 399, 'max_depth': 7, 'learning_rate': 0.10163231970434182, 'subsample': 0.9672271963970651, 'gamma': 2.8279402869601604}


In [ ]:
for trial in study.trials:

    print(f"\nTrial {trial.number}")

    print( f"PR-AUC     : {trial.values[0]:.6f}" )

    print( f"Stabilité  : {trial.values[1]:.6f}" )

    print("\nParamètres :")

    for parameter, value in trial.params.items():

        print( f"  {parameter} = {value}")

In [1]:
!ls /content

sample_data


In [2]:
!git config --global user.name "Chaabmanal2022"
!git config --global user.email "manalchaab2002@gmail.com"

In [3]:
%cd /content

/content


In [4]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [5]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)


In [6]:
%%writefile .gitignore
sample_data/
__pycache__/
*.pyc
.ipynb_checkpoints/
.env

Writing .gitignore


In [7]:
!git checkout -b First-branch

Switched to a new branch 'First-branch'


In [8]:
!git branch

In [10]:
!git remote add origin https://github.com/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection.git

In [11]:
!git remote -v

origin	https://github.com/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection.git (fetch)
origin	https://github.com/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection.git (push)


In [12]:
!git add .

In [13]:
!git status

On branch First-branch

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .config/.last_opt_in_prompt.yaml
	new file:   .config/.last_survey_prompt.yaml
	new file:   .config/.last_update_check.json
	new file:   .config/active_config
	new file:   .config/config_sentinel
	new file:   .config/configurations/config_default
	new file:   .config/default_configs.db
	new file:   .config/gce
	new file:   .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
	new file:   .config/logs/2026.09.16/13.25.37.877256.log
	new file:   .config/logs/2026.09.16/13.26.04.283691.log
	new file:   .config/logs/2026.09.16/13.26.15.129583.log
	new file:   .config/logs/2026.09.16/13.26.17.091876.log
	new file:   .config/logs/2026.09.16/13.26.29.139725.log
	new file:   .config/logs/2026.09.16/13.26.30.459687.log
	new file:   .gitignore



In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import google.colab

# Récupère le nom du notebook et recherche son chemin exact
notebook_name = "Analyse_Dataset.ipynb"
!find /content/drive/MyDrive -name "{notebook_name}" 2>/dev/null

/content/drive/MyDrive/Colab Notebooks/Analyse_Dataset.ipynb
^C


In [16]:
!cp "/content/drive/MyDrive/Colab Notebooks/Pipeline 2 : ADASYN.ipynb" .

In [17]:
!git add "Pipeline 2 : ADASYN.ipynb"

In [18]:
!git status

On branch First-branch

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .config/.last_opt_in_prompt.yaml
	new file:   .config/.last_survey_prompt.yaml
	new file:   .config/.last_update_check.json
	new file:   .config/active_config
	new file:   .config/config_sentinel
	new file:   .config/configurations/config_default
	new file:   .config/default_configs.db
	new file:   .config/gce
	new file:   .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
	new file:   .config/logs/2026.09.16/13.25.37.877256.log
	new file:   .config/logs/2026.09.16/13.26.04.283691.log
	new file:   .config/logs/2026.09.16/13.26.15.129583.log
	new file:   .config/logs/2026.09.16/13.26.17.091876.log
	new file:   .config/logs/2026.09.16/13.26.29.139725.log
	new file:   .config/logs/2026.09.16/13.26.30.459687.log
	new file:   .gitignore
	new file:   Pipeline 2 : ADASYN.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be co

In [19]:
!git commit -m "Add pipeline 2 : ADASYN"

[First-branch (root-commit) 822374e] Add pipeline 2 : ADASYN
 17 files changed, 1026 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.09.16/13.25.37.877256.log
 create mode 100644 .config/logs/2026.09.16/13.26.04.283691.log
 create mode 100644 .config/logs/2026.09.16/13.26.15.129583.log
 create mode 100644 .config/logs/2026.09.16/13.26.17.091876.log
 create mode 100644 .config/logs/2026.09.16/13.26.29.139725.log
 create mode 100644 .config/logs/2026.09.16/13.26.30.459687.log
 create mode 100644 .gitignore
 create mode 1

In [22]:
import getpass

TOKEN = getpass.getpass('Entrez votre GitHub Personal Access Token : ')
REPO_URL = f"https://{TOKEN}@github.com/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection.git"

# Configuration de l'URL distante
!git remote set-url origin {REPO_URL}

Entrez votre GitHub Personal Access Token : ··········


In [23]:
!git push -u origin First-branch

To https://github.com/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection.git
 ! [rejected]        First-branch -> First-branch (fetch first)
error: failed to push some refs to 'https://github.com/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.
